In [3]:
import torch
from torch import nn
import json
import os
import sys

sys.path.append("../")
from data import NiiPoint, CONSTANTS
import torch.nn.functional as F


class NiiKNN(nn.Module):
    
    def __init__(self, k, img_path, label_path, max_num=None):
        super().__init__()
        self.k = k
        self.label_path = label_path
        self.img_path = img_path
        self.max_num = max_num
        
    def forward(self, x: NiiPoint):
        files = os.listdir(self.img_path)
        # read dataset.json to get size
        num_data = len(files)
        dist = torch.zeros(num_data, device="cuda")
        for i,file in enumerate(files):
            print(f"Calculating distance for {file}...", end="\r")
            img = NiiPoint.from_path(os.path.join(self.img_path, file), "cuda")
            diff = x - img
            dist[i] = torch.mean(diff.data**2)**0.5
            if self.max_num is not None and i >= self.max_num:
                break
            
        sorted_idx = torch.argsort(dist)
        x = NiiPoint(torch.zeros_like(x.data, device="cuda"), x.affine, x.header)
        y = NiiPoint(torch.zeros_like(x.data, device="cuda"), x.affine, x.header)
        for i in range(self.k):
            idx = sorted_idx[i]
            file = files[idx]
            label_file = f"{file[:9]}_0001.nii.gz"
            xi = NiiPoint.from_path(os.path.join(self.img_path, file), "cuda")
            yi = NiiPoint.from_path(os.path.join(self.label_path, label_file), "cuda")    
            x = x + xi * (1 / self.k)
            y = y + yi * (1 / self.k)
        return x, y
    

class NiiCKNN(nn.Module):
    
    def __init__(self, k, kernel_size, img_path, label_path, max_num=None, max_batch=None):
        super().__init__()
        self.k = k
        self.kernel_size = kernel_size
        self.label_path = label_path
        self.img_path = img_path
        self.max_num = max_num
        self.max_batch = max_batch
        self.case_ids = [file[:9] for file in os.listdir(self.img_path)]
        self.data_size = len(self.case_ids)
    
    def sliding_l2(self, x_pt: torch.Tensor, pixdims: tuple):
        kernel_size = [2 * int(self.kernel_size[i] / pixdims[i]) + 1 for i in range(3)]
        pad = [int((kernel_size[i] - 1) / 2) for i in range(3)]
        padding = (pad[2], pad[2], pad[1], pad[1], pad[0], pad[0])
        x_pt = x_pt ** 2
        x_pt = F.pad(x_pt.unsqueeze(1), padding, mode='constant', value=0.0)
        weight = torch.ones((1, 1,) + tuple(kernel_size), dtype=x_pt.dtype, device=x_pt.device)
        sliding_sum = F.conv3d(x_pt, weight, stride=1)
        return sliding_sum.squeeze(1)

    
    def forward(self, x: NiiPoint):
        with torch.no_grad():
            files = os.listdir(self.img_path)
            # read dataset.json to get size
            sqdist = torch.full(x.data.shape, torch.inf, device="cuda")                     # track closest block distance
            x_est = torch.zeros_like(x.data, device="cuda")                                   # track closest block image
            y_est = torch.zeros_like(x.data, device="cuda")  
            
            
            for b in range(min(self.max_batch, len(files) // self.max_num)):
                img = torch.zeros((self.max_num, ) + x.data.shape, device="cuda")
                lab = torch.zeros((self.max_num, ) + x.data.shape, device="cuda")
                
                print(f"Processing batch {b + 1}/{min(self.max_batch, len(files) // self.max_num)}...", end="\r")
                for i in range(self.max_num):
                    xi, yi = self.load_case(i + b * self.max_num)
                    img[i, :, :, :] = xi.interpolate_to(x.data.shape).data
                    lab[i, :, :, :] = yi.interpolate_to(x.data.shape).data
                
                
                sqdists_all = self.sliding_l2(
                    x.data[None, :, :, :] - img, 
                    pixdims = x.header.get_zooms()
                    )
                
                sqdist_new, args = torch.min(sqdists_all, dim=0)
                x_new = torch.gather(img, 0, args[None, :, :, :]).squeeze(0)
                y_new = torch.gather(lab, 0, args[None, :, :, :]).squeeze(0)
                
                x_est = torch.where(sqdist_new < sqdist, x_new, x_est)
                y_est = torch.where(sqdist_new < sqdist, y_new, y_est)
                sqdist = torch.min(sqdist, sqdist_new)
                if self.max_num is not None and i >= self.max_num:
                    break
        
            return NiiPoint(x_est, x.affine, x.header), NiiPoint(y_est, x.affine, x.header)

    def load_case(self, idx):
        assert idx < self.data_size, f"Index {idx} out of range for dataset of size {self.data_size}"
        case_id = self.case_ids[idx]
        img_file = f"{case_id}_0000.nii.gz"
        label_file = f"{case_id}_0001.nii.gz"
        img = NiiPoint.from_path(os.path.join(self.img_path, img_file), "cuda")
        label = NiiPoint.from_path(os.path.join(self.label_path, label_file), "cuda")    
        return img, label
        
test_path_img = os.path.join("../", CONSTANTS.NORM_DATA_PATH, f"imagesVl/")
test_path_label = os.path.join("../", CONSTANTS.NORM_DATA_PATH, f"labelsVl/")

train_path_img = os.path.join("../", CONSTANTS.NORM_DATA_PATH, "imagesTr")
train_path_label = os.path.join("../", CONSTANTS.NORM_DATA_PATH, "labelsTr")

target_path_label_cknn = os.path.join("../nnUNet_raw/Predictions_val/cknn/")
target_path_label_knn = os.path.join("../nnUNet_raw/Predictions_val/knn/")
os.makedirs(target_path_label_cknn, exist_ok=True)
os.makedirs(target_path_label_knn, exist_ok=True)

for file_path in os.listdir(test_path_img):
    file_id = file_path[:9]
    img_path = os.path.join(test_path_img, file_path)
    lab_path = os.path.join(test_path_label, f"{file_id}_0001.nii.gz")
    

    x = NiiPoint.from_path(img_path, "cuda")
    y = NiiPoint.from_path(lab_path, "cuda")


    if True:
        model = NiiKNN(3, train_path_img, train_path_label, max_num=100)
        x_pred, y_pred = model(x)
        y_pred.save_to_path(os.path.join(target_path_label_knn, f"{file_id}_0001.nii.gz"))
        print(f"Predicted label for {file_id} saved to {os.path.join(target_path_label_knn, f'{file_id}_0001.nii.gz')}")
    
    
    if False:
        fs = 5
        model = NiiCKNN(3, (fs, fs, fs), train_path_img, train_path_label, max_num=4, max_batch=15)
        x_pred, y_pred = model(x)
        y_pred.save_to_path(os.path.join(target_path_label_cknn, f"{file_id}_0001.nii.gz"))
        print(f"Predicted label for {file_id} saved to {os.path.join(target_path_label_cknn, f'{file_id}_0001.nii.gz')}")

Predicted label for case_0010 saved to ../nnUNet_raw/Predictions_val/knn/case_0010_0001.nii.gz
Predicted label for case_0156 saved to ../nnUNet_raw/Predictions_val/knn/case_0156_0001.nii.gz
Predicted label for case_0136 saved to ../nnUNet_raw/Predictions_val/knn/case_0136_0001.nii.gz
Predicted label for case_0066 saved to ../nnUNet_raw/Predictions_val/knn/case_0066_0001.nii.gz
Predicted label for case_0014 saved to ../nnUNet_raw/Predictions_val/knn/case_0014_0001.nii.gz
Predicted label for case_0089 saved to ../nnUNet_raw/Predictions_val/knn/case_0089_0001.nii.gz
Predicted label for case_0019 saved to ../nnUNet_raw/Predictions_val/knn/case_0019_0001.nii.gz
Predicted label for case_0175 saved to ../nnUNet_raw/Predictions_val/knn/case_0175_0001.nii.gz
Predicted label for case_0016 saved to ../nnUNet_raw/Predictions_val/knn/case_0016_0001.nii.gz
Predicted label for case_0176 saved to ../nnUNet_raw/Predictions_val/knn/case_0176_0001.nii.gz
Predicted label for case_0103 saved to ../nnUNet_r

In [4]:
import matplotlib.pyplot as plt
slices = (0.5, 0.5 ,0.5)
thicks_ncct = (1, 1, 1)
thicks_vess = (10, 10, 10) # Maximum HU range for a layer of thickness 10 voxels 

for case_id in [7, 10, 12]:
    img_path = os.path.join(test_path_img, f"case_{str(case_id).zfill(4)}_0000.nii.gz")
    lab_path = os.path.join(test_path_label, f"case_{str(case_id).zfill(4)}_0001.nii.gz")
    lab_path_pred = os.path.join(target_path_label_cknn, f"case_{str(case_id).zfill(4)}_0001.nii.gz")
    lab_path_pred_knn = os.path.join(target_path_label_knn, f"case_{str(case_id).zfill(4)}_0001.nii.gz")
    
    img_nii = NiiPoint.from_path(img_path, "cuda")
    lab_nii = NiiPoint.from_path(lab_path, "cuda")
    lab_nii_pred = NiiPoint.from_path(lab_path_pred, "cuda")
    lab_nii_pred_knn = NiiPoint.from_path(lab_path_pred_knn, "cuda")

    plt.figure(figsize=(15, 12))
    axes = [
        [plt.subplot(4, 3, i*3 + j + 1) for j in range(3)] for i in range(4)
    ]
    img_nii.plot_slices(slices, thicks_ncct, axes=axes[0])
    lab_nii.plot_slices(slices, thicks_vess, axes=axes[1])
    lab_nii_pred.plot_slices(slices, thicks_vess, axes=axes[2])
    lab_nii_pred_knn.plot_slices(slices, thicks_vess, axes=axes[3])


FileNotFoundError: No such file or no access: '../nnUNet_raw/Predictions_val/cknn/case_0007_0001.nii.gz'